In [2]:
import json
from pathlib import Path
import pandas as pd

files = sorted(Path("../data").glob("reports_*.json"))
reports = pd.DataFrame([r for f in files for r in json.loads(f.read_text())])

print(len(files))
print(reports.shape)
print(reports["service_request_id"].duplicated().sum())
print(reports["requested_datetime"].min(), reports["requested_datetime"].max())

12
(20929, 17)
0
2025-10-01T00:53:24+01:00 2026-09-30T22:41:27+01:00


In [3]:
print(reports.columns.tolist())
print(reports.isna().sum())

['service_name', 'agency_responsible', 'agency_sent_datetime', 'detail', 'requested_datetime', 'updated_datetime', 'interface_used', 'service_code', 'media_url', 'title', 'description', 'long', 'status', 'lat', 'service_request_id', 'comment_count', 'requestor_name']
service_name                0
agency_responsible          0
agency_sent_datetime       41
detail                      0
requested_datetime          0
updated_datetime            0
interface_used              0
service_code                0
media_url               13128
title                       0
description                 0
long                        0
status                      0
lat                         0
service_request_id          0
comment_count            1459
requestor_name          17444
dtype: int64


In [4]:
columns = ["service_request_id", "service_name", "title", "detail", "requested_datetime", "agency_responsible", "interface_used"]
reports = reports[columns]

print(reports.shape)

(20929, 7)


In [5]:
reports["agency_responsible"] = reports["agency_responsible"].apply(lambda a: ", ".join(a.get("recipient", [])))

print(reports["agency_responsible"].nunique())
print(reports["service_name"].nunique())
print(reports["interface_used"].value_counts())

257
57
interface_used
mobile           10726
desktop           4464
Open311           2526
iOS               1381
Android            858
unknown            445
PWA (mobile)       413
PWA (desktop)       71
PWA                 45
Name: count, dtype: int64


In [6]:
reports["text"] = reports["title"].str.strip() + " " + reports["detail"].str.strip()
reports["n_words"] = reports["text"].str.split().str.len()

print(reports["n_words"].describe())
print((reports["n_words"] < 8).sum())
print(reports["text"].duplicated().sum())

count    20929.000000
mean        26.014382
std         27.484012
min          2.000000
25%          9.000000
50%         18.000000
75%         33.000000
max        624.000000
Name: n_words, dtype: float64
3804
3073


In [7]:
short = reports["n_words"] < 8
repeat = reports["text"].duplicated()

print((short & repeat).sum())
print(reports.loc[repeat, "text"].nunique())
print(reports.loc[repeat, "text"].value_counts().head(15))

2497
227
text
Dog fouling problem Dog fouling problem                                                2144
Carcass - Large problem Carcass - Large problem                                         304
Dead fox Dead fox                                                                        73
Dog fouling Dog fouling                                                                  53
Dog poo Dog poo                                                                          28
Carcass - Small problem Carcass - Small problem                                          26
Dog poo Someone hasn't cleaned up after their dog                                        25
Dead fox on pavement Dead fox on pavement                                                18
Dead animal problem Dead animal problem                                                  18
Dead badger Dead badger                                                                  15
Dead fox in road Dead fox in road                                 

In [8]:
same = reports["title"].str.strip().str.lower() == reports["detail"].str.strip().str.lower()

print(same.sum())
print((same | short | repeat).sum())
print(reports.loc[same, "interface_used"].value_counts())

3510
4953
interface_used
Open311         2521
mobile           483
desktop          330
iOS               78
Android           74
PWA (mobile)      15
unknown            9
Name: count, dtype: int64


In [9]:
keep = ~(same | short | repeat)
corpus = reports[keep].copy()

print(corpus.shape)
print(corpus["interface_used"].value_counts())
print(corpus["service_name"].value_counts().head(10))

(15976, 9)
interface_used
mobile           9451
desktop          3931
iOS              1090
Android           675
unknown           401
PWA (mobile)      315
PWA (desktop)      67
PWA                41
Open311             5
Name: count, dtype: int64
service_name
Dog fouling                                3312
Dead Animal                                2342
Dog Fouling                                2172
Dead animal                                1981
Dead animals                               1010
Dead animal (off the road)                  690
Street cleaning - Dead animal on street     599
Dog Poo/Fouling                             542
Street cleaning - Dog mess                  441
Dog Waste Clearance (Street)                393
Name: count, dtype: int64


In [11]:
import nltk

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Rebecca/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Rebecca/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/Rebecca/nltk_data...


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english")) | {"please", "council", "report", "reported", "problem"}
lemmatizer = WordNetLemmatizer()

def clean(text):
    tokens = word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(w for w in words if w not in stop_words)

print(clean("The dead foxes were reported to the council last week, but nobody has removed them. Please help!"))